# State-of-the-art guardrail stack lab

Build a replaceable guardrail stack for a synthetic multi-tenant benefits assistant. The lab uses Pydantic and frozen representative SDK responses, so it runs offline without credentials.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from stack_lab import (
    Policy, Scenario, StackPlan, decide, evaluate, load_catalog,
    normalize_response, release_gate, select_stack, trace_event,
)
FIXTURES = Path('fixtures')
load = lambda name: json.loads((FIXTURES / name).read_text())

## 1. Select against requirements

The catalog contains dated capability claims and primary-source links. The greedy selector is deliberately explainable; `StackPlan` performs the important check by requiring coverage, deployment compatibility, and an application-owned control.

In [ ]:
catalog = load_catalog(FIXTURES / 'tool_catalog.json')
scenario = Scenario.model_validate(load('scenario.json'))
plan = StackPlan(selected=select_stack(catalog, scenario), scenario=scenario)
[(tool.tool_id, tool.kind, sorted(tool.capabilities & scenario.required_capabilities)) for tool in plan.selected]

## 2. Normalize incompatible SDK responses

Application policy should consume a stable signal rather than binding directly to each product's response fields. This makes replacement testable and keeps vendor output from becoming an authorization decision.

In [ ]:
cases = load('evaluation_cases.json')
first_signals = [normalize_response(item['source'], item['response']) for item in cases[1]['signals']]
[signal.model_dump() for signal in first_signals]

## 3. Make failure behavior explicit

The policy fails closed when a required detector is unavailable before a write. A read may continue in a degraded mode with a reason code. Change this choice only after documenting harm, reversibility, and the fallback experience.

In [ ]:
policy = Policy(
    injection_threshold=0.8, content_threshold=0.8,
    transform_pii=True, fail_closed_for={'write'},
    version='support-guardrails/3.2',
)
outcomes = []
for case in cases:
    signals = [normalize_response(item['source'], item['response']) for item in case['signals']]
    outcomes.append((case['case_id'], decide(signals, policy, case['operation'])))
[(case_id, outcome.decision.value, outcome.reason_codes, outcome.degraded) for case_id, outcome in outcomes]

## 4. Evaluate the composed policy

The release gate sees exact decisions, false negatives and positives with denominators, and degraded cases. Raising the injection threshold demonstrates why a configuration change needs the same regression suite as a code change.

In [ ]:
baseline = evaluate(cases, policy)
regressed = evaluate(cases, policy.model_copy(update={'injection_threshold': 0.99}))
{'baseline': (baseline, release_gate(baseline)), 'regressed': (regressed, release_gate(regressed))}

## 5. Preserve privacy-minimized evidence

The trace record uses an OpenInference guardrail span kind and keeps the versioned decision without copying a raw benefits request into telemetry.

In [ ]:
trace_event(outcomes[1][1], 'trace-synthetic-42')

## Exercises

1. Require self-hosted deployment only and inspect which capabilities become uncovered.
2. Add a multilingual false-positive case and report the language slice.
3. Replace one frozen response shape while keeping `GuardSignal` and every policy test unchanged.
4. Change the degraded read from allow to escalate and document the availability trade-off.